# Step 2: Text Generation for Test-Class Descriptions

## Test Classes (16)
Descriptions are generated for the novel test classes.


## 2.1 Import Required Libraries


In [ ]:
import os
import json
import time
import random
from pathlib import Path
from typing import List, Dict
import requests
from tqdm import tqdm
import numpy as np

print("Libraries imported successfully!")

## 2.2 Configure the Qwen Max API

Provide the API key through the QIANWEN_API_KEY environment variable.


In [ ]:
QIANWEN_API_KEY = os.environ.get("QIANWEN_API_KEY", "")

QIANWEN_API_URL = "https://dashscope.aliyuncs.com/api/v1/services/aigc/text-generation/generation"
MODEL_NAME = "qwen-max"
TEMPERATURE = 0.7
TOP_P = 0.9
MAX_TOKENS = 70
MAX_LABELS_PER_SAMPLE = 5
OUTPUT_DIR = os.environ.get("DEPF_DATA_DIR", "coco2014_data")

if not QIANWEN_API_KEY:
    print("  WARNING: Set the QIANWEN_API_KEY environment variable before generating descriptions.")
else:
    print(" API Key configured")

print(f"Model: {MODEL_NAME}")
print(f"Max tokens: {MAX_TOKENS}")
print(f"Max labels per sample: {MAX_LABELS_PER_SAMPLE}")
print(f"Output directory: {OUTPUT_DIR}")
print("\nText descriptions are generated for the novel and validation classes.")

## 2.3 Define Class Sets


In [ ]:
NOVEL_LABELS_COCO = [
    'bicycle', 'boat', 'stop sign', 'bird', 'backpack', 'frisbee', 'snowboard', 'surfboard',
    'cup', 'fork', 'spoon', 'broccoli', 'chair', 'keyboard', 'microwave', 'vase'
]

VAL_LABELS_COCO = [
    'cow', 'dining table', 'zebra', 'sandwich', 'bear', 'toaster', 'person', 'laptop',
    'bed', 'teddy bear', 'baseball bat', 'skis'
]

BASE_LABELS_COCO = [
    'hair drier', 'clock', 'wine glass', 'book', 'cake', 'tie', 'motorcycle', 'sheep',
    'bottle', 'giraffe', 'cell phone', 'suitcase', 'remote', 'bench', 'mouse', 'carrot',
    'banana', 'train', 'sports ball', 'toothbrush', 'fire hydrant', 'airplane', 'tv',
    'bus', 'refrigerator', 'couch', 'knife', 'toilet', 'elephant', 'truck', 'parking meter',
    'car', 'potted plant', 'kite', 'skateboard', 'orange', 'horse', 'cat', 'tennis racket',
    'bowl', 'scissors', 'baseball glove', 'apple', 'traffic light', 'handbag', 'donut',
    'dog', 'hot dog', 'oven', 'umbrella', 'sink', 'pizza'
]

print(f"Base classes: {len(BASE_LABELS_COCO)}")
print(f"Validation classes: {len(VAL_LABELS_COCO)}")
print(f"Novel classes: {len(NOVEL_LABELS_COCO)}")
print(f"Total: {len(BASE_LABELS_COCO) + len(VAL_LABELS_COCO) + len(NOVEL_LABELS_COCO)}")

## 2.4 Load ImageNet-Style Templates


In [ ]:
IMAGENET_TEMPLATES = [
    "a bad photo of a {}.",
    "a photo of many {}.",
    "a sculpture of a {}.",
    "a photo of the hard to see {}.",
    "a low resolution photo of the {}.",
    "a rendering of a {}.",
    "graffiti of a {}.",
    "a bad photo of the {}.",
    "a cropped photo of the {}.",
    "a tattoo of a {}.",
    "the embroidered {}.",
    "a photo of a hard to see {}.",
    "a bright photo of a {}.",
    "a photo of a clean {}.",
    "a photo of a dirty {}.",
    "a dark photo of the {}.",
    "a drawing of a {}.",
    "a photo of my {}.",
    "the plastic {}.",
    "a photo of the cool {}.",
    "a close-up photo of a {}.",
    "a black and white photo of the {}.",
    "a painting of the {}.",
    "a painting of a {}.",
    "a pixelated photo of the {}.",
    "a sculpture of the {}.",
    "a bright photo of the {}.",
    "a cropped photo of a {}.",
    "a plastic {}.",
    "a photo of the dirty {}.",
    "a jpeg corrupted photo of a {}.",
    "a blurry photo of the {}.",
    "a photo of the {}.",
    "a good photo of the {}.",
    "a rendering of the {}.",
    "a {} in a video game.",
    "a photo of one {}.",
    "a doodle of a {}.",
    "a close-up photo of the {}.",
    "a photo of a {}.",
    "the origami {}.",
    "the {} in a video game.",
    "a sketch of a {}.",
    "a doodle of the {}.",
    "a origami {}.",
    "a low resolution photo of a {}.",
    "the toy {}.",
    "a rendition of the {}.",
    "a photo of the clean {}.",
    "a photo of a large {}.",
    "a rendition of a {}.",
    "a photo of a nice {}.",
    "a photo of a weird {}.",
    "a blurry photo of a {}.",
    "a cartoon {}.",
    "art of a {}.",
    "a sketch of the {}.",
    "a embroidered {}.",
    "a pixelated photo of a {}.",
    "itap of the {}.",
    "a jpeg corrupted photo of the {}.",
    "a good photo of a {}.",
    "a plushie {}.",
    "a photo of the nice {}.",
    "a photo of the small {}.",
    "a photo of the weird {}.",
    "the cartoon {}.",
    "art of the {}.",
    "a drawing of the {}.",
    "a photo of the large {}.",
    "a black and white photo of a {}.",
    "the plushie {}.",
    "a dark photo of a {}.",
    "itap of a {}.",
    "graffiti of the {}.",
    "a toy {}.",
    "itap of my {}.",
    "a photo of a cool {}.",
    "a photo of a small {}.",
    "a tattoo of the {}.",
]

print(f"ImageNet templates loaded: {len(IMAGENET_TEMPLATES)}")
print("\nSample templates:")
for i, template in enumerate(IMAGENET_TEMPLATES[:5], 1):
    print(f"  {i}. {template}")

## 2.5 Implement the Qwen API Call


In [ ]:
def generate_description(labels: List[str], api_key: str) -> str:
    """Generate one text description for a list of class labels."""
    prompt = f"Please make a brief sentence to describe a photo that contains {', '.join(labels)}."
    
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }
    
    payload = {
        "model": MODEL_NAME,
        "input": {
            "messages": [
                {
                    "role": "system",
                    "content": "You are a helpful assistant that provides brief descriptions of photos containing objects."
                },
                {
                    "role": "user",
                    "content": prompt
                }
            ]
        },
        "parameters": {
            "temperature": TEMPERATURE,
            "top_p": TOP_P,
            "max_tokens": MAX_TOKENS,
            "result_format": "message"
        }
    }
    
    max_retries = 3
    for attempt in range(max_retries):
        try:
            response = requests.post(
                QIANWEN_API_URL,
                headers=headers,
                json=payload,
                timeout=60
            )
            
            response.raise_for_status()
            result = response.json()
            
            if 'output' in result and 'choices' in result['output']:
                content = result['output']['choices'][0]['message']['content']
                return content.strip()
            else:
                print("  Unexpected API response format")
                return ""
                
        except requests.exceptions.Timeout:
            print(f"  Timeout on attempt {attempt + 1}/{max_retries}")
            if attempt < max_retries - 1:
                time.sleep(2 ** attempt)
                
        except requests.exceptions.RequestException as e:
            print(f" Request error on attempt {attempt + 1}/{max_retries}")
            if attempt < max_retries - 1:
                time.sleep(2 ** attempt)
                
        except Exception as e:
            print(" Unexpected generation error")
            break
    
    return ""

if QIANWEN_API_KEY:
    print("Testing API with sample labels ['bicycle', 'person']...")
    test_description = generate_description(['bicycle', 'person'], QIANWEN_API_KEY)
    
    if test_description:
        print("\n API connection successful!")
        print(f"Test description: {test_description}")
    else:
        print("\n API connection failed.")
else:
    print("  API key not configured. Skipping test.")

## 2.6 Implement Label Sampling


In [ ]:
def sample_labels(label_set: List[str], max_labels: int = 5) -> List[str]:
    """Randomly sample between one and max_labels class names."""
    num_labels = random.randint(1, max_labels)
    return random.sample(label_set, num_labels)

print("Testing label sampling...")
for i in range(5):
    sampled = sample_labels(NOVEL_LABELS_COCO, max_labels=5)
    print(f"  Sample {i+1}: {sampled} ({len(sampled)} labels)")

## 2.7 Generate Test Descriptions: Multi-Label Combinations

Each generated description combines one to five class labels. NUM_NOVEL_SAMPLES controls the number of test descriptions.


In [ ]:
def generate_data(
    label_set: List[str],
    api_key: str,
    num_samples: int = 10000,
    max_labels: int = 5,
    save_interval: int = 500
) -> List[Dict]:
    """Generate descriptions for sampled label combinations and periodically save progress."""
    data = []
    
    print("="*70)
    print(f"GENERATING {num_samples} DESCRIPTIONS")
    print("="*70)
    print(f"Label set: {len(label_set)} categories")
    print(f"Max labels per sample: {max_labels}")
    print(f"Estimated API calls: {num_samples}")
    print(f"Estimated time: {num_samples * 2 / 60:.1f} minutes")
    print("="*70)
    
    for i in tqdm(range(num_samples), desc="Generating descriptions"):
        labels = sample_labels(label_set, max_labels)
        
        description = generate_description(labels, api_key)
        
        if description:
            data.append({'labels': labels, 'description': description})
        
        if (i + 1) % save_interval == 0:
            intermediate_file = os.path.join(
                OUTPUT_DIR,
                f"novel_descriptions_intermediate_{i+1}.json"
            )
            os.makedirs(OUTPUT_DIR, exist_ok=True)
            with open(intermediate_file, 'w', encoding='utf-8') as f:
                json.dump(data, f, ensure_ascii=False, indent=2)
            print(f"\n Saved intermediate: {intermediate_file} ({len(data)} samples)")
        
        time.sleep(0.5)
    
    print("\n" + "="*70)
    print(f" Generated {len(data)} descriptions successfully!")
    print("="*70)
    
    return data

NUM_NOVEL_SAMPLES = 10000
if QIANWEN_API_KEY:
    print("\n" + "="*70)
    print("READY TO GENERATE NOVEL (TEST) SET DESCRIPTIONS")
    print("="*70)
    print(f"Categories: {len(NOVEL_LABELS_COCO)}")
    print(f"Samples to generate: {NUM_NOVEL_SAMPLES}")
    print(f"Max labels per sample: {MAX_LABELS_PER_SAMPLE}")
    print(f"Estimated time: {NUM_NOVEL_SAMPLES * 0.5 / 60:.1f} minutes")
    print("\n  This will consume API quota.")
    print("\nTo proceed, run the cell below.")
else:
    print("  API key not configured.")

## 2.8 Generate Test-Set Descriptions


In [ ]:
if QIANWEN_API_KEY:
    novel_data = generate_data(
        label_set=NOVEL_LABELS_COCO,
        api_key=QIANWEN_API_KEY,
        num_samples=NUM_NOVEL_SAMPLES,
        max_labels=MAX_LABELS_PER_SAMPLE,
        save_interval=500
    )
    
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    novel_file = os.path.join(OUTPUT_DIR, 'novel_descriptions_10k.json')
    with open(novel_file, 'w', encoding='utf-8') as f:
        json.dump(novel_data, f, ensure_ascii=False, indent=2)
    
    print(f"\n Novel descriptions saved to: {novel_file}")
    print(f"File size: {os.path.getsize(novel_file) / 1024 / 1024:.2f} MB")
else:
    print(" API key not configured. Skipping generation.")

## 2.9 Generate Validation-Set Descriptions


In [ ]:
# if QIANWEN_API_KEY and 'novel_data' in locals():
#     print("\n" + "="*70)
#     print("GENERATING VALIDATION SET DESCRIPTIONS")
#     print("="*70)
    
#     val_data = generate_data(
#         label_set=VAL_LABELS_COCO,
#         api_key=QIANWEN_API_KEY,
#         num_samples=NUM_VAL_SAMPLES,
#         max_labels=MAX_LABELS_PER_SAMPLE,
#         save_interval=500
#     )
    
#     val_file = os.path.join(OUTPUT_DIR, 'val_descriptions_5k.json')
#     with open(val_file, 'w', encoding='utf-8') as f:
#         json.dump(val_data, f, ensure_ascii=False, indent=2)
    
#     print(f"\n Validation descriptions saved to: {val_file}")
#     print(f"File size: {os.path.getsize(val_file) / 1024 / 1024:.2f} MB")
# else:
#     print("  API key not configured or novel data not generated yet.")

## 2.10 Apply ImageNet Template Augmentation

ImageNet templates provide additional descriptions for each class.


In [ ]:
def apply_imagenet_templates(categories: List[str], templates: List[str]) -> Dict[str, List[str]]:
    """Create template-based descriptions for each class."""
    enhanced_descriptions = {}
    
    for category in categories:
        descriptions = [template.format(category) for template in templates]
        enhanced_descriptions[category] = descriptions
    
    return enhanced_descriptions

print("Applying ImageNet templates to novel classes...")
novel_template_enhanced = apply_imagenet_templates(NOVEL_LABELS_COCO, IMAGENET_TEMPLATES)

print("\nNovel class template enhancement:")
for cat, descs in list(novel_template_enhanced.items())[:3]:
    print(f"  {cat:<15} {len(descs):>3} descriptions")
print(f"  ... ({len(novel_template_enhanced)} total classes)")

novel_template_file = os.path.join(OUTPUT_DIR, 'novel_template_enhanced.json')
os.makedirs(OUTPUT_DIR, exist_ok=True)
with open(novel_template_file, 'w', encoding='utf-8') as f:
    json.dump(novel_template_enhanced, f, ensure_ascii=False, indent=2)

print(f"\n Novel template-enhanced descriptions saved to: {novel_template_file}")

In [ ]:
# Optional validation-set template generation.
# val_template_enhanced = apply_imagenet_templates(VAL_LABELS_COCO, IMAGENET_TEMPLATES)
# print("\nValidation class template enhancement:")
# for cat, descs in val_template_enhanced.items():
#     print(f"  {cat:<15} {len(descs):>3} descriptions")
# print(f"  ... ({len(val_template_enhanced)} total classes)")
# val_template_file = os.path.join(OUTPUT_DIR, "val_template_enhanced.json")
# with open(val_template_file, "w", encoding="utf-8") as f:
#     json.dump(val_template_enhanced, f, ensure_ascii=False, indent=2)


## 2.11 Merge Descriptions from Both Strategies


In [ ]:
def merge_descriptions(
    api_generated_data: List[Dict],
    template_enhanced_data: Dict[str, List[str]],
    categories: List[str]
) -> Dict[str, Dict]:
    """Merge API-generated and template-based descriptions by class."""
    merged = {}
    
    for category in categories:
        merged[category] = {
            'api_descriptions': [],            'template_descriptions': template_enhanced_data.get(category, []),
            'all_descriptions': []
        }
    
    for item in api_generated_data:
        labels = item['labels']
        description = item['description']
        
        for label in labels:
            if label in merged:
                merged[label]['api_descriptions'].append(description)
    
    for category in categories:
        all_descs = (
            merged[category]['api_descriptions'] + 
            merged[category]['template_descriptions']
        )
        merged[category]['all_descriptions'] = all_descs
        merged[category]['num_api_descriptions'] = len(merged[category]['api_descriptions'])
        merged[category]['num_template_descriptions'] = len(merged[category]['template_descriptions'])
        merged[category]['num_total_descriptions'] = len(all_descs)
    
    return merged

if 'novel_data' in locals():
    print("Merging novel class descriptions...")
    merged_novel = merge_descriptions(
        novel_data,
        novel_template_enhanced,
        NOVEL_LABELS_COCO
    )
    
    print("\nNovel class description statistics:")
    print(f"{'Category':<15} {'API':>6} {'Template':>8} {'Total':>6}")
    print("-" * 40)
    for cat in NOVEL_LABELS_COCO:
        stats = merged_novel[cat]
        print(f"{cat:<15} {stats['num_api_descriptions']:>6} {stats['num_template_descriptions']:>8} {stats['num_total_descriptions']:>6}")
    
    merged_novel_file = os.path.join(OUTPUT_DIR, 'novel_descriptions_merged.json')
    with open(merged_novel_file, 'w', encoding='utf-8') as f:
        json.dump(merged_novel, f, ensure_ascii=False, indent=2)
    print(f"\n Merged novel descriptions saved to: {merged_novel_file}")
else:
    print("  Novel data not generated yet.")

In [ ]:
# if 'val_data' in locals():
#     print("Merging validation class descriptions...")
#     merged_val = merge_descriptions(
#         val_data,
#         val_template_enhanced,
#         VAL_LABELS_COCO
#     )
    
#     print("\nValidation class description statistics:")
#     print(f"{'Category':<15} {'API':>6} {'Template':>8} {'Total':>6}")
#     print("-" * 40)
#     for cat in VAL_LABELS_COCO:
#         stats = merged_val[cat]
#         print(f"{cat:<15} {stats['num_api_descriptions']:>6} {stats['num_template_descriptions']:>8} {stats['num_total_descriptions']:>6}")
    
#     merged_val_file = os.path.join(OUTPUT_DIR, 'val_descriptions_merged.json')
#     with open(merged_val_file, 'w', encoding='utf-8') as f:
#         json.dump(merged_val, f, ensure_ascii=False, indent=2)
#     print(f"\n Merged validation descriptions saved to: {merged_val_file}")
# else:
#     print("  Validation data not generated yet.")

## 2.12 Inspect Generated Description Examples


In [ ]:
def display_sample_descriptions(merged_data: Dict, category: str, num_samples: int = 5):
    """Display a selection of descriptions for one class."""
    if category not in merged_data:
        print(f"  Category '{category}' not found.")
        return
    
    print(f"\n{'='*70}")
    print(f"Descriptions for: {category.upper()}")
    print(f"{'='*70}")
    
    data = merged_data[category]
    
    print(f"\nAPI-generated descriptions (first {num_samples}):")
    for i, desc in enumerate(data['api_descriptions'][:num_samples], 1):
        print(f"  {i}. {desc}")
    
    print(f"\nTemplate-generated descriptions (first {num_samples}):")
    for i, desc in enumerate(data['template_descriptions'][:num_samples], 1):
        print(f"  {i}. {desc}")
    
    print(f"\nTotal: {data['num_total_descriptions']} descriptions")

if 'merged_novel' in locals():
    display_sample_descriptions(merged_novel, 'bicycle', num_samples=3)
    display_sample_descriptions(merged_novel, 'cup', num_samples=3)
else:
    print("  Merged novel data not available.")

## 2.13 Generate a Summary Report


In [ ]:
def generate_final_report():
    """Summarize text-generation counts and output files."""
    report = f"""
{'='*70}
TEXT GENERATION SUMMARY REPORT
{'='*70}

1. GENERATION STRATEGY
{'-'*70}
   Strategy 1: Multi-label combination descriptions (API-generated)
   Strategy 2: ImageNet-style template descriptions (80 templates)
   Final: Merged both strategies for maximum coverage

2. GENERATION PARAMETERS
{'-'*70}
   Model: {MODEL_NAME}
   Temperature: {TEMPERATURE}
   Top-P: {TOP_P}
   Max Tokens: {MAX_TOKENS}
   ImageNet Templates: {len(IMAGENET_TEMPLATES)}

3. CATEGORIES
{'-'*70}
"""
    
    if 'novel_data' in locals():
        report += f"   Novel Classes (Test): {len(NOVEL_LABELS_COCO)}\n"
        report += f"   API samples generated: {len(novel_data)}\n"
    
    # if 'val_data' in locals():
    #     report += f"   Validation Classes: {len(VAL_LABELS_COCO)}\n"
    #     report += f"   API samples generated: {len(val_data)}\n"
    
    report += f"""

4. OUTPUT FILES
{'-'*70}
"""
    
    output_files = []
    
    if 'novel_data' in locals():
        output_files.extend([
            ('novel_descriptions_10k.json', 'API-generated novel descriptions'),
            ('novel_template_enhanced.json', 'Template-enhanced novel descriptions'),
            ('novel_descriptions_merged.json', 'Merged novel descriptions (RECOMMENDED)')
        ])
    
    if 'val_data' in locals():
        output_files.extend([
            ('val_descriptions_5k.json', 'API-generated validation descriptions'),
            ('val_template_enhanced.json', 'Template-enhanced validation descriptions'),
            ('val_descriptions_merged.json', 'Merged validation descriptions (RECOMMENDED)')
        ])
    
    for filename, description in output_files:
        filepath = os.path.join(OUTPUT_DIR, filename)
        if os.path.exists(filepath):
            size_mb = os.path.getsize(filepath) / 1024 / 1024
            report += f"    {filename}\n"
            report += f"     ({description}, {size_mb:.2f} MB)\n"
        else:
            report += f"    {filename} (not found)\n"
    
    report += f"""

5. DESCRIPTION STATISTICS
{'-'*70}
"""
    for cat in NOVEL_LABELS_COCO[:5]:
        stats = merged_novel.get(cat, {})
        report += f"    {cat}: {stats.get('num_total_descriptions', 0)} descriptions\n"
    
    # Validation-set statistics are included when validation descriptions are available.
    if 'val_data' in locals():
        for cat in VAL_LABELS_COCO[:5]:
            stats = merged_val.get(cat, {})
            report += f"    {cat}: {stats.get('num_total_descriptions', 0)} descriptions\n"
    
    report += f"\n"
    report += f"The report summarizes generated descriptions by class.\n"
    report += f"\n"
    report += f"\n"
    report += f"\n"
    report += f"\n"
    report += f"""

6. USAGE RECOMMENDATION
{'-'*70}
   For CoMC training/inference:
   - Use 'novel_descriptions_merged.json' for test set
   - Use 'val_descriptions_merged.json' for validation set
   - These contain both API-generated and template-enhanced descriptions
   - Each category has 80+ descriptions for robustness

{'='*70}
Text generation completed successfully!
{'='*70}
"""
    
    return report

if 'novel_data' in locals() or 'val_data' in locals():
    report = generate_final_report()
    print(report)
    
    report_file = os.path.join(OUTPUT_DIR, 'text_generation_report.txt')
    with open(report_file, 'w', encoding='utf-8') as f:
        f.write(report)
    print(f"\n Report saved to: {report_file}")
else:
    print("  No data generated yet. Cannot generate report.")

## Summary

### Output Files
- novel_descriptions_merged.json: descriptions for the 16 test classes
- val_descriptions_merged.json: descriptions for the 12 validation classes
- text_generation_report.txt: generation report
